# Longitudinal COVID-19 Proteomics Analysis

**Secondary analysis author:** Janessa Reed, MS  
**Original study/data:** Gisby et al. (2021), *Longitudinal proteomic profiling of dialysis patients with COVID-19 reveals markers of severity and predictors of death*. [Original paper](https://doi.org/10.7554/eLife.64827) · [Original GitHub repository](https://github.com/jackgisby/longitudinal_olink_proteomics)

**What is original here?** The Python data auditing, exploratory QC, visualization, and statistical screening—not the clinical study, patient recruitment, sample collection, or laboratory measurements. AI assisted with code and documentation; all findings remain exploratory and require human review.


### AI Assistance Disclosure

ChatGPT was used to assist with Python code development, debugging, statistical methodology, data visualization, workflow organization, and documentation.

The underlying proteomics and clinical data were collected and published by Gisby et al. (2021), not generated by AI.

This project represents an independent, AI-assisted secondary analysis of published biomedical research. All statistical findings are exploratory and require appropriate validation before being interpreted as scientific conclusions.


**How to use:** Run cells from top to bottom in VS Code's Jupyter notebook interface. Each section explains *what*, *why*, and *how to check it*. This notebook replaces the need to run the `.py` file for interactive learning. It does **not** claim raw-instrument data processing, GWAS, or clinical validation.

## 0. Folder setup and software

Save this notebook in a `notebooks/` folder inside your project. Keep the two original CSV files **locally** in `data/` and do not upload them simply to demonstrate the secondary analysis.

Install requirements in a VS Code terminal if needed: `python -m pip install pandas numpy matplotlib seaborn scikit-learn statsmodels jupyter ipykernel`.

**Why:** Working-directory problems caused the earlier `FileNotFoundError`. This setup checks the project root explicitly.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
DATA, OUT, FIG = ROOT / "data", ROOT / "outputs", ROOT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
for file in [DATA / "plasma_npx_level.csv", DATA / "plasma_sample_level.csv"]:
    if not file.is_file():
        raise FileNotFoundError(f"Cannot find {file}. Current working directory: {cwd}")
print("Project root:", ROOT)

def save(df, name):
    df.to_csv(OUT / name, index=False)

## 1. Load and reconcile the biological data

**Question:** Do samples and patients agree between the assay measurements and clinical metadata? One row of the protein dataset represents a **sample–panel–assay** measurement; one metadata row represents a sample.

**Decision rule:** Stop with an error if keys or patient-to-sample links contradict each other. Do not silently drop discrepancies.

In [ ]:
p = pd.read_csv(DATA/'plasma_npx_level.csv')
m = pd.read_csv(DATA/'plasma_sample_level.csv')
key = ['SampleID','Panel','Assay']
for name, df, cols in [
    ('proteomics',p,key+['Individual_ID','NPX']),
    ('metadata',m,['SampleID','Individual_ID','Plate_ID','Case_Control','WHO_Severity_Peak','Time_From_First_Swab'])
]:
    absent = set(cols) - set(df.columns)
    if absent: raise ValueError(f'{name}: missing {absent}')
if p.duplicated(key).any(): raise ValueError('Nonunique sample-panel-assay keys; reconcile before continuing')
if m.SampleID.duplicated().any(): raise ValueError('Nonunique metadata SampleID')
if p[key].isna().any().any() or m.SampleID.isna().any(): raise ValueError('Missing essential IDs')
p['NPX'] = pd.to_numeric(p.NPX, errors='raise')
if not np.isfinite(p.NPX).all(): raise ValueError('Nonfinite NPX values')
link = p[['SampleID','Individual_ID']].drop_duplicates().merge(m[['SampleID','Individual_ID']],on='SampleID',how='outer',indicator=True,validate='one_to_one',suffixes=('_protein','_metadata'))
if not link._merge.eq('both').all() or not link.Individual_ID_protein.eq(link.Individual_ID_metadata).all(): raise ValueError('Sample to patient metadata reconciliation failed')

In [ ]:
print('Measurement rows:',len(p),'Metadata samples:',len(m))
print('Unique patients:',m.Individual_ID.nunique())
print('Panels:',p.Panel.nunique(),'Distinct panel–assays:',len(p[['Panel','Assay']].drop_duplicates()))
display(p.head(3),m.head(3))

## 2. Assay completeness and missingness

**Question:** Which sample–assay combinations are structurally absent, even if the `NPX` column has no `NaN` entries?

**Important assumption:** The expected assay reference below comes from all panel–assay combinations **observed in these files**, not an independent laboratory manifest. Missingness does not automatically mean a laboratory failure or low protein abundance.

In [ ]:
# Assay completeness reference comes from assays observed in this particular file,
# not from an independent instrument manifest.
ref = p[['Panel','Assay']].drop_duplicates()
expected = m[['SampleID']].merge(ref,how='cross')
observed = p[key].drop_duplicates()
absence = expected.merge(observed,on=key,how='left',indicator=True,validate='one_to_one')
absence = absence.loc[absence._merge.eq('left_only'),key].merge(m[['SampleID','Plate_ID','Case_Control']],on='SampleID',validate='many_to_one')
save(absence,'structurally_absent_assays.csv')
save(p.isna().sum().rename_axis('column').reset_index(name='missing'), 'proteomics_column_missingness.csv')
save(m.isna().sum().rename_axis('column').reset_index(name='missing'), 'metadata_column_missingness.csv')

In [ ]:
print('Structurally absent assay combinations:', len(absence))
print('Absences by panel:')
display(absence.Panel.value_counts())
print('Samples missing most assays:')
display(absence.groupby('SampleID').size().sort_values(ascending=False).head(15))

## 3. Sample-panel and assay-level quality control

A sample with an entire absent panel is different from a sample with one missing assay. **NPX** is log2-scaled *relative* expression; it is not an absolute concentration, and NPX magnitudes across different proteins should not be interpreted as directly comparable protein quantities.

**Check:** We expect 307 × 5 = 1,535 potential sample-panel entries from the five observed panels; three fully absent panels imply 1,532 observed entries.

In [ ]:
# Sample-panel QC (do not compare NPX across individual assays as absolute concentration).
joined = p.merge(m[['SampleID','Plate_ID','Case_Control','Individual_ID','WHO_Severity_Peak','Time_From_First_Swab']],on='SampleID',how='left',validate='many_to_one',suffixes=('','_meta'))
sample_panel = joined.groupby(['SampleID','Panel','Plate_ID','Case_Control'],as_index=False).agg(median_npx=('NPX','median'),n_assays=('Assay','nunique'))
save(sample_panel,'sample_panel_qc.csv')

# Long-format assay-level QC, no imputation.
assay_qc = joined.groupby(['Panel','Assay'],as_index=False).agg(n_samples=('SampleID','nunique'),median_npx=('NPX','median'),sd_npx=('NPX','std'))
assay_qc['coverage_pct'] = assay_qc.n_samples / m.SampleID.nunique()*100
save(assay_qc,'assay_qc.csv')

In [ ]:
print('Observed sample-panel rows:', len(sample_panel))
display(assay_qc.sort_values('coverage_pct').head(10))
# Simple exploratory plot: sample-panel medians, not protein-specific inference
sns.catplot(data=sample_panel, x='Plate_ID', y='median_npx', col='Panel', col_wrap=3,
            kind='box', sharey=False, height=3)
plt.show()

## 4. PCA — a visual quality-control tool

**What PCA does:** It reduces many correlated protein variables to a few axes that capture patterns of variation. Here we select only assays measured for **every** sample, standardize each assay, then calculate two principal components.

**Limit:** This exclusion can make the PCA subset unrepresentative of the entire 436-assay panel. Apparent clustering by plate cannot alone establish a *technical* batch effect.

In [ ]:
# PCA restricted to assay columns measured for all samples: complete-case assays.
# This avoids constructing imputed values and is a QC visualization only.
p['feature'] = p['Panel'].astype(str)+'::'+p['Assay'].astype(str)
mat = p.pivot(index='SampleID',columns='feature',values='NPX')
complete = mat.loc[:,mat.notna().all(axis=0)]
if complete.shape[1] < 3: raise ValueError('Not enough complete assays for PCA')
z = StandardScaler().fit_transform(complete)
pca = PCA(n_components=2)
pcs = pca.fit_transform(z)
pca_df = pd.DataFrame({'SampleID':complete.index,'PC1':pcs[:,0],'PC2':pcs[:,1]}).merge(m[['SampleID','Plate_ID','Case_Control']],on='SampleID',validate='one_to_one')
save(pca_df,'pca_coordinates.csv')
fig,ax=plt.subplots(figsize=(9,6)); sns.scatterplot(data=pca_df,x='PC1',y='PC2',hue='Plate_ID',style='Case_Control',alpha=.8,ax=ax)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)'); ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA of complete-case Olink NPX assays (descriptive QC)'); fig.tight_layout(); fig.savefig(FIG/'pca_plate_covid.png',dpi=170); plt.close(fig)

In [ ]:
print('Samples × complete assays:', complete.shape)
print('PC1 variance %:',round(pca.explained_variance_ratio_[0]*100,2))
print('PC2 variance %:',round(pca.explained_variance_ratio_[1]*100,2))
display(pca_df.head())
display(__import__('IPython').display.Image(filename=str(FIG/'pca_plate_covid.png')))

## 5. Exploratory time association for each panel–assay

The code screens assay-specific associations with time **among COVID-positive samples** using ordinary least squares with patient-clustered standard errors and plate indicators. It is **not the same** as our earlier IL-6 random-intercept mixed model. This explains why coefficients and p-values can differ between the two.

**Why FDR?** Testing hundreds of proteins creates false-positive risk. Benjamini–Hochberg adjusts p-values across successfully fitted assays. FDR < 0.05 indicates *screening candidates*, not confirmed clinical biomarkers.

**Limitations:** Patient-clustered uncertainty does not solve informative follow-up, time-varying confounding, nonlinear trajectories, or possible between-patient versus within-patient differences.

In [ ]:
# Longitudinal association: separate exploratory OLS for every assay, restricted
# to COVID-POSITIVE observations. Inference uses PATIENT-CLUSTERED ROBUST SEs
# to address repeated measures; NOT equivalent to a mixed-effects model.
# A fixed plate term is descriptive, NOT proof of removing technical batch effects.
positive = joined.loc[(joined.Case_Control=='POSITIVE') & joined.Time_From_First_Swab.notna()].copy()
positive['time_days']=pd.to_numeric(positive.Time_From_First_Swab,errors='coerce')
rows=[]
for (panel,assay),g in positive.groupby(['Panel','Assay'],sort=True):
    g=g.dropna(subset=['time_days','NPX','Plate_ID','Individual_ID'])
    n=g.Individual_ID.nunique()
    if n < 20 or len(g) < 40 or g.time_days.nunique()<5:
        rows.append({'Panel':panel,'Assay':assay,'status':'insufficient_data','n_observations':len(g),'n_patients':n})
        continue
    try:
        fit=smf.ols('NPX ~ time_days + C(Plate_ID)',data=g).fit(cov_type='cluster',cov_kwds={'groups':g.Individual_ID,'use_correction':True})
        b=float(fit.params['time_days']); ci=fit.conf_int().loc['time_days']
        rows.append({'Panel':panel,'Assay':assay,'status':'ok','n_observations':len(g),'n_patients':n,'beta_npx_per_day':b,'ci_low':float(ci.iloc[0]),'ci_high':float(ci.iloc[1]),'p_value':float(fit.pvalues['time_days'])})
    except Exception as exc:
        rows.append({'Panel':panel,'Assay':assay,'status':'model_error','n_observations':len(g),'n_patients':n,'error':str(exc)[:180]})
result=pd.DataFrame(rows)
result['fdr_bh']=np.nan
valid=result.status.eq('ok') & result.p_value.notna()
if valid.any(): result.loc[valid,'fdr_bh']=multipletests(result.loc[valid,'p_value'],method='fdr_bh')[1]
result=result.sort_values(['status','fdr_bh','p_value'],na_position='last')
save(result,'all_assay_longitudinal_cluster_robust.csv')

In [ ]:
print('Successful assay screens:',valid.sum())
print('Exploratory FDR < 0.05:',(result.fdr_bh < .05).sum())
display(result[['Panel','Assay','status','n_observations','n_patients','beta_npx_per_day','p_value','fdr_bh']].head(15))
il6_screen = result.query("Assay == 'IL6' and Panel == 'In'")
display(il6_screen)

## 6. Metadata consistency and run summary

**Patient-level warning:** `WHO_Severity_Peak` is a patient's peak disease outcome, not a measurement that changes at every time point. Be alert to duplicated patient-level outcomes in a sample-level table.

**Correction from the prior script:** The earlier `run_summary.csv` used the name `positive_measurements` for **all** 307 IL-6 records. This notebook uses `positive_il6_measurements` to count the actual COVID-positive subset (256 in the supplied data). The previous GitHub summary needs this correction if published.

In [ ]:
# Sample-level severity metadata is patient-constant? Examine before modeling severity.
severity_var = m.loc[m.Case_Control.eq('POSITIVE')].groupby('Individual_ID')['WHO_Severity_Peak'].nunique()
severity_consistent = bool(severity_var.le(1).all())

In [ ]:
summary = {
 'proteomics_rows':len(p), 'metadata_samples':len(m),
 'patients':m.Individual_ID.nunique(),
 'assay_definitions':len(ref), 'sample_assay_absences':len(absence),
 'fully_absent_sample_panels':int((absence.groupby(['SampleID','Panel']).size().reset_index(name='absent').merge(ref.groupby('Panel').size().reset_index(name='expected'),on='Panel').eval('absent == expected')).sum()),
 'pca_complete_assays':complete.shape[1],
 'pca_pc1_variance_pct':round(pca.explained_variance_ratio_[0]*100,2),
 'n_modeled_assays':int(valid.sum()),
 'n_fdr_lt_005':int((result.fdr_bh < .05).sum()),
 'positive_patients':positive.Individual_ID.nunique(),
 'positive_il6_measurements':int(((positive.Assay=='IL6') & (positive.Panel=='In')).sum()),
 'peak_severity_constant_within_patient':severity_consistent,
}
summary_df = pd.Series(summary).rename_axis('metric').reset_index(name='value')
summary_df.to_csv(OUT / 'run_summary.csv',index=False)
display(summary_df)

## 7. Reproducibility checkpoint

These assertions compare our outputs to **independently observed input counts from this exact research-data download**, not universal truths about any version of the study. If an input changes, inspect the cause rather than editing assertions just to make them pass.

The statistical outputs are *not fully validated* by these count checks. A stronger analysis still needs model diagnostics, repeatability checks, independent statistics review, and severity-specific analyses.

In [ ]:
assert len(p) == 133577
assert len(m) == 307
assert m.Individual_ID.nunique() == 105
assert len(ref) == 436
assert len(absence) == 275
assert len(sample_panel) == 1532
assert complete.shape == (307, 261)
assert int(((positive.Assay=='IL6') & (positive.Panel=='In')).sum()) == 256
assert len(result) == 436
assert len(list(OUT.glob('*.csv'))) >= 8
print('Reference data-integrity and run-summary checks passed.')

## 8. Interpretation, attribution, and interviewer talking points

**What I did:** Validated 133,577 published Olink measurement records, reconciled samples and metadata, audited 275 structurally absent sample–assay combinations, assessed sample-panel QC, made PCA figures using 261 complete assays, and screened 436 assays for exploratory time associations with patient-clustered uncertainty.

**What I did not do:** Collect patients' samples; process raw mass-spectrometry or sequencing files; prove a laboratory plate effect; establish causality; or validate biomarkers for clinical use.

**Questions to rehearse:**
1. Why is zero missing NPX not the same as complete assay coverage?
2. Why is a patient not the same analytical unit as a sample?
3. What does log2 NPX measure, and why shouldn't different proteins' NPX magnitudes be directly compared?
4. Why can a patient-clustered OLS screen disagree with a mixed-effects model?
5. Why use FDR adjustment when screening 436 proteins?
6. Why is a PCA plate cluster not proof of a batch artifact?

**Ethics and credit:** Gisby et al. created and published the original research data. This notebook is an independent, AI-assisted secondary analysis; cite the original article and its repository. Check original data reuse terms before redistributing any raw data. Keep the code and results open to inspection and the analysis claims appropriately qualified.